
# Grain Robustness Q1 — Notebook 01, versión 2
## Auditoría, trazabilidad y congelamiento de Dry Bean e INIAP Rice

Esta versión evita trabajar directamente sobre el sistema de archivos de
Google Drive durante el análisis. Google Drive se utiliza únicamente para:

1. localizar o recibir los archivos Excel;
2. copiar el paquete final de resultados.

Todo el procesamiento se realiza primero en `/content`, que es más estable y
rápido que el sistema FUSE de Google Drive.

### Qué hace este notebook

1. Monta y verifica Google Drive.
2. Localiza los dos archivos Excel en Drive o en `/content`.
3. Copia los archivos a un espacio local temporal.
4. Calcula huellas SHA-256 de los archivos fuente.
5. Audita dimensiones, tipos, valores faltantes, infinitos y clases.
6. Detecta duplicados exactos y posibles conflictos de etiqueta.
7. Verifica relaciones algebraicas conocidas.
8. Calcula correlaciones y dimensionalidad efectiva.
9. Añade identificadores reproducibles y grupos de duplicados.
10. Congela los datos en CSV y Parquet.
11. Genera tablas, figuras, manifiestos y un ZIP.
12. Sincroniza los resultados finales con Google Drive mediante reintentos.

### Principios de esta etapa

- No se eliminan observaciones.
- No se imputan valores.
- No se entrenan modelos.
- No se crean todavía particiones de entrenamiento, calibración o prueba.
- Los duplicados se conservan, pero reciben el mismo `duplicate_group_id`.
- El notebook debe ejecutarse con **CPU**. No requiere GPU.


## 1. Preparar el entorno de Colab

In [ ]:

!pip -q install openpyxl pyarrow


In [ ]:

from google.colab import drive
from pathlib import Path
import os
import time

MOUNT_POINT = Path('/content/drive')

# Fuerza un montaje nuevo para evitar conexiones FUSE obsoletas.
drive.mount(str(MOUNT_POINT), force_remount=True)

MY_DRIVE = MOUNT_POINT / 'MyDrive'

# Esperar hasta que Drive esté realmente disponible.
last_error = None
for attempt in range(1, 7):
    try:
        if not MY_DRIVE.exists():
            raise FileNotFoundError(f'No existe {MY_DRIVE}')

        probe_path = MY_DRIVE / '.grain_robustness_drive_probe.txt'
        probe_path.write_text(
            f'Drive write test: attempt {attempt}',
            encoding='utf-8',
        )
        probe_path.unlink(missing_ok=True)

        print('Google Drive montado y con escritura verificada.')
        break

    except Exception as error:
        last_error = error
        print(
            f'Intento {attempt}/6: Drive aún no está estable: '
            f'{type(error).__name__}: {error}'
        )
        time.sleep(5)

else:
    raise RuntimeError(
        'Google Drive no quedó disponible después de seis intentos. '
        'Reinicia el entorno de ejecución de Colab, vuelve a ejecutar '
        'esta celda y autoriza nuevamente el acceso. '
        f'Último error: {last_error}'
    )


## 2. Importaciones y configuración

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import shutil
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 20260803

# Trabajo local: evita errores de conexión de Google Drive durante mkdir,
# escritura de CSV/Parquet, creación de figuras y compresión.
PROJECT_ROOT = Path('/content/Grain_Robustness_Q1_runtime')

# Destino persistente en Google Drive. Se utilizará solo al final.
DRIVE_PROJECT_ROOT = Path(
    '/content/drive/MyDrive/Grain_Robustness_Q1'
)

RAW_DIR = PROJECT_ROOT / '01_data_raw'
FROZEN_DIR = PROJECT_ROOT / '02_data_frozen'
PROTOCOL_DIR = PROJECT_ROOT / '03_protocol'
RESULTS_DIR = PROJECT_ROOT / '07_results' / '01_AUDIT_FREEZE'
FIGURES_DIR = PROJECT_ROOT / '08_figures' / '01_AUDIT_FREEZE'
TABLES_DIR = PROJECT_ROOT / '09_tables' / '01_AUDIT_FREEZE'

for directory in [
    RAW_DIR,
    FROZEN_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'dry_bean': {
        'filenames': [
            'Dry_Bean_Dataset.xlsx',
            'Dry Bean Dataset.xlsx',
        ],
        'standard_raw_name': 'Dry_Bean_Dataset.xlsx',
        'sheet': 'Dry_Beans_Dataset',
        'target': 'Class',
        'expected_rows': 13611,
        'expected_classes': 7,
    },
    'iniap_rice': {
        'filenames': [
            'Granos de Arroz.xlsx',
            'Granos_de_Arroz_INIAP.xlsx',
            'Granos de Arroz INIAP.xlsx',
        ],
        'standard_raw_name': 'Granos_de_Arroz_INIAP.xlsx',
        'sheet': 'Sheet1',
        'target': 'CLASS',
        'expected_rows': 9200,
        'expected_classes': 3,
    },
}

print('Espacio de trabajo local:', PROJECT_ROOT)
print('Destino persistente en Drive:', DRIVE_PROJECT_ROOT)
print('Runtime:', platform.platform())
print('Python:', sys.version.split()[0])
print('CPU disponibles:', os.cpu_count())


## 3. Localizar y copiar los archivos fuente

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def safe_stat_mtime(path):
    try:
        return path.stat().st_mtime
    except OSError:
        return 0.0


def locate_file(candidate_names):
    # 1) Buscar primero en el espacio local oficial.
    for candidate_name in candidate_names:
        candidate = RAW_DIR / candidate_name
        if candidate.exists():
            return candidate

    # 2) Buscar archivos cargados directamente al runtime de Colab.
    content_root = Path('/content')
    local_matches = []
    for candidate_name in candidate_names:
        local_matches.extend(content_root.glob(candidate_name))

    local_matches = [
        path for path in local_matches
        if path.is_file()
    ]
    if local_matches:
        return sorted(
            set(local_matches),
            key=safe_stat_mtime,
            reverse=True,
        )[0]

    # 3) Buscar en Mi unidad. Se elige la copia modificada más recientemente.
    my_drive = Path('/content/drive/MyDrive')
    matches = []

    for candidate_name in candidate_names:
        try:
            matches.extend(my_drive.rglob(candidate_name))
        except (ConnectionAbortedError, OSError) as error:
            print(
                'Advertencia durante la búsqueda en Drive:',
                type(error).__name__,
                error,
            )
            time.sleep(3)

    matches = [
        path for path in matches
        if path.is_file()
    ]

    if not matches:
        return None

    matches = sorted(
        set(matches),
        key=safe_stat_mtime,
        reverse=True,
    )

    if len(matches) > 1:
        print('ADVERTENCIA: se encontraron varias copias:')
        for match in matches:
            print(' -', match)
        print('Se utilizará la copia más reciente:', matches[0])

    return matches[0]


def copy_to_local_with_retry(source, destination, attempts=6):
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            shutil.copy2(source, destination)

            if not destination.exists():
                raise IOError(
                    f'La copia local no existe: {destination}'
                )

            if destination.stat().st_size != source.stat().st_size:
                raise IOError(
                    'El tamaño de la copia local no coincide con el origen.'
                )

            return destination

        except (
            ConnectionAbortedError,
            ConnectionResetError,
            OSError,
            IOError,
        ) as error:
            last_error = error
            print(
                f'Intento de copia {attempt}/{attempts} falló: '
                f'{type(error).__name__}: {error}'
            )
            time.sleep(5)

    raise RuntimeError(
        f'No fue posible copiar {source} a {destination}. '
        f'Último error: {last_error}'
    )


source_manifest_rows = []
source_paths = {}

for dataset_id, config in EXPECTED.items():
    located = locate_file(config['filenames'])

    if located is None:
        raise FileNotFoundError(
            f"No se encontró el archivo de {dataset_id}. "
            "Puedes cargarlo directamente en el panel Archivos de Colab "
            "o guardarlo en cualquier carpeta de Mi unidad con uno de "
            f"estos nombres: {config['filenames']}"
        )

    destination = RAW_DIR / config['standard_raw_name']

    if located.resolve() != destination.resolve():
        copy_to_local_with_retry(located, destination)

    source_paths[dataset_id] = destination

    source_manifest_rows.append({
        'dataset_id': dataset_id,
        'original_path_found': str(located),
        'local_path_used': str(destination),
        'source_filename': destination.name,
        'size_bytes': destination.stat().st_size,
        'modified_utc': datetime.fromtimestamp(
            destination.stat().st_mtime,
            tz=timezone.utc,
        ).isoformat(),
        'sha256': sha256_file(destination),
    })

source_manifest = pd.DataFrame(source_manifest_rows)
source_manifest.to_csv(
    RESULTS_DIR / '01_source_file_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

display(source_manifest)


## 4. Cargar los archivos y comprobar su estructura

In [ ]:

datasets_raw = {}
load_rows = []

for dataset_id, config in EXPECTED.items():
    path = source_paths[dataset_id]
    excel = pd.ExcelFile(path)

    if config['sheet'] not in excel.sheet_names:
        raise ValueError(
            f"La hoja esperada '{config['sheet']}' no existe en {path.name}. "
            f"Hojas disponibles: {excel.sheet_names}"
        )

    dataframe = pd.read_excel(path, sheet_name=config['sheet'])
    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]

    if config['target'] not in dataframe.columns:
        raise ValueError(
            f"No se encontró la variable objetivo '{config['target']}' "
            f"en {dataset_id}. Columnas: {dataframe.columns.tolist()}"
        )

    datasets_raw[dataset_id] = dataframe

    load_rows.append({
        'dataset_id': dataset_id,
        'sheet': config['sheet'],
        'rows_observed': len(dataframe),
        'columns_observed': dataframe.shape[1],
        'classes_observed': dataframe[config['target']].nunique(dropna=False),
        'expected_rows': config['expected_rows'],
        'expected_classes': config['expected_classes'],
        'row_count_matches_expected': len(dataframe) == config['expected_rows'],
        'class_count_matches_expected': (
            dataframe[config['target']].nunique(dropna=False)
            == config['expected_classes']
        ),
    })

load_validation = pd.DataFrame(load_rows)
load_validation.to_csv(
    RESULTS_DIR / '02_load_validation.csv',
    index=False,
    encoding='utf-8-sig',
)

display(load_validation)

for dataset_id, dataframe in datasets_raw.items():
    print('\n', '=' * 80)
    print(dataset_id, dataframe.shape)
    display(dataframe.head())


## 5. Funciones de auditoría y trazabilidad por observación

In [ ]:

def canonical_value(value):
    if pd.isna(value):
        return '<NA>'

    if isinstance(value, (np.integer, int)):
        return str(int(value))

    if isinstance(value, (np.floating, float)):
        if not np.isfinite(value):
            return str(value)
        return format(float(value), '.17g')

    return str(value).strip()


def row_sha256(dataframe, columns):
    hashes = []
    for row in dataframe[columns].itertuples(index=False, name=None):
        canonical = '\x1f'.join(canonical_value(value) for value in row)
        hashes.append(
            hashlib.sha256(canonical.encode('utf-8')).hexdigest()
        )
    return pd.Series(hashes, index=dataframe.index, dtype='string')


def make_record_ids(prefix, n_rows):
    width = max(6, len(str(n_rows)))
    return [
        f'{prefix}_{index:0{width}d}'
        for index in range(1, n_rows + 1)
    ]


def audit_dataset(dataset_id, dataframe, target_column):
    feature_columns = [
        column for column in dataframe.columns
        if column != target_column
    ]
    numeric_columns = dataframe[
        feature_columns
    ].select_dtypes(include=[np.number]).columns.tolist()

    exact_duplicate_mask = dataframe.duplicated(
        subset=dataframe.columns.tolist(),
        keep=False,
    )
    feature_duplicate_mask = dataframe.duplicated(
        subset=feature_columns,
        keep=False,
    )

    missing_cells = int(dataframe.isna().sum().sum())

    if numeric_columns:
        numeric_values = dataframe[numeric_columns].to_numpy(dtype=float)
        infinite_cells = int(np.isinf(numeric_values).sum())
    else:
        infinite_cells = 0

    audit = {
        'dataset_id': dataset_id,
        'rows': len(dataframe),
        'columns_total': dataframe.shape[1],
        'features': len(feature_columns),
        'numeric_features': len(numeric_columns),
        'classes': dataframe[target_column].nunique(dropna=False),
        'missing_cells': missing_cells,
        'infinite_cells': infinite_cells,
        'exact_duplicate_rows_involved': int(exact_duplicate_mask.sum()),
        'exact_duplicate_groups': int(
            dataframe.loc[exact_duplicate_mask]
            .groupby(dataframe.columns.tolist(), dropna=False)
            .ngroups
        ) if exact_duplicate_mask.any() else 0,
        'feature_duplicate_rows_involved': int(feature_duplicate_mask.sum()),
    }

    return audit, feature_columns, numeric_columns


audit_rows = []
dataset_columns = {}

for dataset_id, dataframe in datasets_raw.items():
    target = EXPECTED[dataset_id]['target']
    audit, feature_columns, numeric_columns = audit_dataset(
        dataset_id,
        dataframe,
        target,
    )
    audit_rows.append(audit)
    dataset_columns[dataset_id] = {
        'target': target,
        'features': feature_columns,
        'numeric': numeric_columns,
    }

audit_summary = pd.DataFrame(audit_rows)
audit_summary.to_csv(
    RESULTS_DIR / '03_dataset_audit_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

display(audit_summary)


## 6. Distribución de clases

In [ ]:

class_distribution_parts = []

for dataset_id, dataframe in datasets_raw.items():
    target = dataset_columns[dataset_id]['target']

    counts = (
        dataframe[target]
        .astype('string')
        .value_counts(dropna=False)
        .rename_axis('class_label')
        .reset_index(name='n')
    )
    counts['proportion'] = counts['n'] / len(dataframe)
    counts.insert(0, 'dataset_id', dataset_id)
    class_distribution_parts.append(counts)

class_distribution = pd.concat(
    class_distribution_parts,
    ignore_index=True,
)

class_distribution.to_csv(
    TABLES_DIR / 'Table_class_distribution.csv',
    index=False,
    encoding='utf-8-sig',
)

display(class_distribution)

for dataset_id in datasets_raw:
    plot_data = class_distribution[
        class_distribution['dataset_id'] == dataset_id
    ].sort_values('n', ascending=False)

    figure, axis = plt.subplots(figsize=(9, 5))
    axis.bar(plot_data['class_label'], plot_data['n'])
    axis.set_xlabel('Clase')
    axis.set_ylabel('Número de observaciones')
    axis.set_title(f'Distribución de clases — {dataset_id}')
    axis.tick_params(axis='x', rotation=35)
    axis.grid(True, axis='y', alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f'class_distribution_{dataset_id}.png',
        dpi=300,
        bbox_inches='tight',
    )
    figure.savefig(
        FIGURES_DIR / f'class_distribution_{dataset_id}.pdf',
        bbox_inches='tight',
    )
    plt.show()


## 7. Duplicados exactos y conflictos de etiqueta

In [ ]:

duplicate_reports = []
label_conflict_reports = []

for dataset_id, dataframe in datasets_raw.items():
    target = dataset_columns[dataset_id]['target']
    features = dataset_columns[dataset_id]['features']

    working = dataframe.copy()
    working['_original_row'] = np.arange(2, len(working) + 2)

    # Duplicados exactos: mismas características y misma etiqueta.
    exact_mask = working.duplicated(
        subset=dataframe.columns.tolist(),
        keep=False,
    )

    if exact_mask.any():
        exact_rows = working.loc[exact_mask].copy()
        exact_rows['_duplicate_count'] = (
            exact_rows.groupby(
                dataframe.columns.tolist(),
                dropna=False,
            )[target]
            .transform('size')
        )
        exact_rows.insert(0, 'dataset_id', dataset_id)
        duplicate_reports.append(exact_rows)

    # Conflicto potencial: mismas características, etiquetas diferentes.
    feature_group = (
        working.groupby(features, dropna=False)[target]
        .agg(
            number_of_rows='size',
            number_of_labels='nunique',
            labels=lambda values: ' | '.join(
                sorted(
                    set(values.astype(str))
                )
            ),
        )
        .reset_index()
    )

    conflicts = feature_group[
        feature_group['number_of_labels'] > 1
    ].copy()

    if not conflicts.empty:
        conflicts.insert(0, 'dataset_id', dataset_id)
        label_conflict_reports.append(conflicts)

if duplicate_reports:
    exact_duplicates = pd.concat(
        duplicate_reports,
        ignore_index=True,
    )
else:
    exact_duplicates = pd.DataFrame(
        columns=['dataset_id', '_original_row']
    )

if label_conflict_reports:
    label_conflicts = pd.concat(
        label_conflict_reports,
        ignore_index=True,
    )
else:
    label_conflicts = pd.DataFrame(
        columns=[
            'dataset_id',
            'number_of_rows',
            'number_of_labels',
            'labels',
        ]
    )

exact_duplicates.to_csv(
    RESULTS_DIR / '04_exact_duplicate_rows.csv',
    index=False,
    encoding='utf-8-sig',
)
label_conflicts.to_csv(
    RESULTS_DIR / '05_feature_duplicate_label_conflicts.csv',
    index=False,
    encoding='utf-8-sig',
)

print('Filas involucradas en duplicados exactos:', len(exact_duplicates))
print('Grupos con características idénticas y etiquetas distintas:', len(label_conflicts))

display(exact_duplicates.head(20))
display(label_conflicts.head(20))



## 8. Verificación de relaciones algebraicas

Las verificaciones se realizan sobre relaciones documentables a partir de las
propias columnas. No se interpreta todavía importancia biológica.

### Dry Bean

Se comprueban:

\[
\mathrm{AspectRatio}=\frac{\mathrm{MajorAxisLength}}{\mathrm{MinorAxisLength}}
\]

\[
e=\sqrt{1-\left(\frac{\mathrm{MinorAxisLength}}{\mathrm{MajorAxisLength}}\right)^2}
\]

\[
\mathrm{EquivalentDiameter}=\sqrt{\frac{4\,\mathrm{Area}}{\pi}}
\]

\[
\mathrm{Solidity}=\frac{\mathrm{Area}}{\mathrm{ConvexArea}}
\]

\[
\mathrm{Roundness}=\frac{4\pi\,\mathrm{Area}}{\mathrm{Perimeter}^2}
\]

\[
\mathrm{Compactness}=\frac{\mathrm{EquivalentDiameter}}{\mathrm{MajorAxisLength}}
\]

\[
\mathrm{ShapeFactor1}=\frac{\mathrm{MajorAxisLength}}{\mathrm{Area}}
\]

\[
\mathrm{ShapeFactor3}=\mathrm{Compactness}^2
\]

### INIAP Rice

Se comprueba la excentricidad a partir de los ejes mayor y menor.


In [ ]:

def relationship_report(
    dataset_id,
    relationship_name,
    observed,
    reconstructed,
    absolute_tolerance,
    relative_tolerance,
):
    observed = np.asarray(observed, dtype=float)
    reconstructed = np.asarray(reconstructed, dtype=float)

    finite = np.isfinite(observed) & np.isfinite(reconstructed)
    observed = observed[finite]
    reconstructed = reconstructed[finite]

    absolute_error = np.abs(observed - reconstructed)
    scale = np.maximum(np.abs(observed), np.finfo(float).eps)
    relative_error = absolute_error / scale

    within = np.isclose(
        observed,
        reconstructed,
        atol=absolute_tolerance,
        rtol=relative_tolerance,
    )

    return {
        'dataset_id': dataset_id,
        'relationship': relationship_name,
        'n_valid': int(len(observed)),
        'max_absolute_error': float(absolute_error.max()),
        'mean_absolute_error': float(absolute_error.mean()),
        'median_absolute_error': float(np.median(absolute_error)),
        'max_relative_error': float(relative_error.max()),
        'proportion_within_tolerance': float(within.mean()),
        'absolute_tolerance': absolute_tolerance,
        'relative_tolerance': relative_tolerance,
        'passes_all_rows': bool(within.all()),
    }


relationship_rows = []

bean = datasets_raw['dry_bean'].copy()

bean_relationships = {
    'AspectRation = MajorAxisLength / MinorAxisLength': (
        bean['AspectRation'],
        bean['MajorAxisLength'] / bean['MinorAxisLength'],
        1e-12,
        1e-12,
    ),
    'Eccentricity from major and minor axes': (
        bean['Eccentricity'],
        np.sqrt(
            np.maximum(
                0.0,
                1.0 - (
                    bean['MinorAxisLength']
                    / bean['MajorAxisLength']
                ) ** 2,
            )
        ),
        1e-12,
        1e-12,
    ),
    'EquivDiameter from Area': (
        bean['EquivDiameter'],
        np.sqrt(4.0 * bean['Area'] / np.pi),
        1e-10,
        1e-12,
    ),
    'Solidity = Area / ConvexArea': (
        bean['Solidity'],
        bean['Area'] / bean['ConvexArea'],
        1e-12,
        1e-12,
    ),
    'roundness = 4*pi*Area / Perimeter^2': (
        bean['roundness'],
        4.0 * np.pi * bean['Area'] / (bean['Perimeter'] ** 2),
        1e-12,
        1e-12,
    ),
    'Compactness = EquivDiameter / MajorAxisLength': (
        bean['Compactness'],
        bean['EquivDiameter'] / bean['MajorAxisLength'],
        1e-12,
        1e-12,
    ),
    'ShapeFactor1 = MajorAxisLength / Area': (
        bean['ShapeFactor1'],
        bean['MajorAxisLength'] / bean['Area'],
        1e-12,
        1e-12,
    ),
    'ShapeFactor3 = Compactness^2': (
        bean['ShapeFactor3'],
        bean['Compactness'] ** 2,
        1e-12,
        1e-12,
    ),
}

for relationship_name, (
    observed,
    reconstructed,
    atol,
    rtol,
) in bean_relationships.items():
    relationship_rows.append(
        relationship_report(
            'dry_bean',
            relationship_name,
            observed,
            reconstructed,
            atol,
            rtol,
        )
    )

rice = datasets_raw['iniap_rice'].copy()

rice_eccentricity = np.sqrt(
    np.maximum(
        0.0,
        1.0 - (
            rice['MINOR_AXIS']
            / rice['MAJOR_AXIS']
        ) ** 2,
    )
)

relationship_rows.append(
    relationship_report(
        'iniap_rice',
        'ECCENTRICITY from MAJOR_AXIS and MINOR_AXIS',
        rice['ECCENTRICITY'],
        rice_eccentricity,
        1e-4,
        1e-4,
    )
)

algebraic_relationships = pd.DataFrame(relationship_rows)
algebraic_relationships.to_csv(
    TABLES_DIR / 'Table_algebraic_relationships.csv',
    index=False,
    encoding='utf-8-sig',
)

display(algebraic_relationships)


## 9. Resumen descriptivo de variables numéricas

In [ ]:

numeric_summary_parts = []

for dataset_id, dataframe in datasets_raw.items():
    numeric_columns = dataset_columns[dataset_id]['numeric']

    summary = (
        dataframe[numeric_columns]
        .describe(
            percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )
        .T
        .reset_index()
        .rename(columns={'index': 'feature'})
    )

    summary.insert(0, 'dataset_id', dataset_id)
    summary['missing'] = [
        int(dataframe[column].isna().sum())
        for column in summary['feature']
    ]
    summary['unique_values'] = [
        int(dataframe[column].nunique(dropna=True))
        for column in summary['feature']
    ]

    numeric_summary_parts.append(summary)

numeric_summary = pd.concat(
    numeric_summary_parts,
    ignore_index=True,
)

numeric_summary.to_csv(
    TABLES_DIR / 'Table_numeric_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

display(numeric_summary)


## 10. Correlación y dimensionalidad efectiva

In [ ]:

pca_rows = []
high_correlation_rows = []

for dataset_id, dataframe in datasets_raw.items():
    numeric_columns = dataset_columns[dataset_id]['numeric']
    numeric_data = dataframe[numeric_columns].astype(float)

    correlation = numeric_data.corr(method='pearson')
    correlation.to_csv(
        RESULTS_DIR / f'correlation_matrix_{dataset_id}.csv',
        encoding='utf-8-sig',
    )

    # Pares de correlación absoluta alta, sin repetir la diagonal.
    for first_index, first_feature in enumerate(numeric_columns):
        for second_index in range(first_index + 1, len(numeric_columns)):
            second_feature = numeric_columns[second_index]
            coefficient = correlation.loc[
                first_feature, second_feature
            ]

            if abs(coefficient) >= 0.95:
                high_correlation_rows.append({
                    'dataset_id': dataset_id,
                    'feature_1': first_feature,
                    'feature_2': second_feature,
                    'pearson_r': float(coefficient),
                    'absolute_r': float(abs(coefficient)),
                })

    # PCA solo como auditoría de dimensionalidad efectiva.
    standardized = StandardScaler().fit_transform(numeric_data)
    pca = PCA(random_state=RANDOM_SEED)
    pca.fit(standardized)
    cumulative = np.cumsum(pca.explained_variance_ratio_)

    pca_rows.append({
        'dataset_id': dataset_id,
        'numeric_features': len(numeric_columns),
        'components_for_90_percent': int(
            np.searchsorted(cumulative, 0.90) + 1
        ),
        'components_for_95_percent': int(
            np.searchsorted(cumulative, 0.95) + 1
        ),
        'components_for_97_5_percent': int(
            np.searchsorted(cumulative, 0.975) + 1
        ),
        'components_for_99_percent': int(
            np.searchsorted(cumulative, 0.99) + 1
        ),
        'variance_first_component': float(
            pca.explained_variance_ratio_[0]
        ),
        'variance_first_three_components': float(
            pca.explained_variance_ratio_[:3].sum()
        ),
        'variance_first_five_components': float(
            pca.explained_variance_ratio_[:5].sum()
        ),
    })

    figure, axis = plt.subplots(figsize=(9, 7))
    image = axis.imshow(
        correlation.to_numpy(),
        vmin=-1.0,
        vmax=1.0,
        aspect='auto',
    )
    axis.set_xticks(np.arange(len(numeric_columns)))
    axis.set_xticklabels(
        numeric_columns,
        rotation=60,
        ha='right',
        fontsize=8,
    )
    axis.set_yticks(np.arange(len(numeric_columns)))
    axis.set_yticklabels(numeric_columns, fontsize=8)
    axis.set_title(f'Matriz de correlación — {dataset_id}')

    figure.colorbar(
        image,
        ax=axis,
        label='Correlación de Pearson',
    )
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f'correlation_heatmap_{dataset_id}.png',
        dpi=300,
        bbox_inches='tight',
    )
    figure.savefig(
        FIGURES_DIR / f'correlation_heatmap_{dataset_id}.pdf',
        bbox_inches='tight',
    )
    plt.show()

pca_summary = pd.DataFrame(pca_rows)
high_correlations = pd.DataFrame(high_correlation_rows).sort_values(
    ['dataset_id', 'absolute_r'],
    ascending=[True, False],
)

pca_summary.to_csv(
    RESULTS_DIR / '06_pca_effective_dimension.csv',
    index=False,
    encoding='utf-8-sig',
)
high_correlations.to_csv(
    RESULTS_DIR / '07_high_correlations_abs_ge_0_95.csv',
    index=False,
    encoding='utf-8-sig',
)

display(pca_summary)
display(high_correlations.head(50))


## 11. Crear identificadores y congelar los datasets

In [ ]:

frozen_datasets = {}
frozen_manifest_rows = []

prefixes = {
    'dry_bean': 'DB',
    'iniap_rice': 'IR',
}

for dataset_id, dataframe in datasets_raw.items():
    target = dataset_columns[dataset_id]['target']
    features = dataset_columns[dataset_id]['features']

    frozen = dataframe.copy()

    # La huella de características permite mantener juntas copias exactas
    # aunque aparezcan en distintas filas.
    feature_hash = row_sha256(frozen, features)
    full_row_hash = row_sha256(
        frozen,
        features + [target],
    )

    metadata = pd.DataFrame({
        'dataset_id': dataset_id,
        'record_id': make_record_ids(
            prefixes[dataset_id],
            len(frozen),
        ),
        'source_excel_row': np.arange(2, len(frozen) + 2),
        'feature_fingerprint_sha256': feature_hash,
        'full_row_fingerprint_sha256': full_row_hash,
        'duplicate_group_id': (
            prefixes[dataset_id]
            + '_G_'
            + feature_hash.str.slice(0, 16)
        ),
        'exact_duplicate_count': (
            feature_hash.groupby(feature_hash)
            .transform('size')
            .astype(int)
        ),
    })

    frozen = pd.concat(
        [
            metadata.reset_index(drop=True),
            frozen.reset_index(drop=True),
        ],
        axis=1,
    )

    csv_path = FROZEN_DIR / f'{dataset_id}_frozen_v1.csv'
    parquet_path = FROZEN_DIR / f'{dataset_id}_frozen_v1.parquet'

    frozen.to_csv(
        csv_path,
        index=False,
        encoding='utf-8-sig',
    )
    frozen.to_parquet(
        parquet_path,
        index=False,
    )

    frozen_datasets[dataset_id] = frozen

    frozen_manifest_rows.extend([
        {
            'dataset_id': dataset_id,
            'format': 'csv',
            'path': str(csv_path),
            'rows': len(frozen),
            'columns': frozen.shape[1],
            'size_bytes': csv_path.stat().st_size,
            'sha256': sha256_file(csv_path),
        },
        {
            'dataset_id': dataset_id,
            'format': 'parquet',
            'path': str(parquet_path),
            'rows': len(frozen),
            'columns': frozen.shape[1],
            'size_bytes': parquet_path.stat().st_size,
            'sha256': sha256_file(parquet_path),
        },
    ])

frozen_manifest = pd.DataFrame(frozen_manifest_rows)
frozen_manifest.to_csv(
    RESULTS_DIR / '08_frozen_file_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

display(frozen_manifest)


## 12. Verificación posterior al congelamiento

In [ ]:

freeze_validation_rows = []

for dataset_id, frozen_original in frozen_datasets.items():
    csv_path = FROZEN_DIR / f'{dataset_id}_frozen_v1.csv'
    parquet_path = FROZEN_DIR / f'{dataset_id}_frozen_v1.parquet'

    csv_reload = pd.read_csv(csv_path)
    parquet_reload = pd.read_parquet(parquet_path)

    expected_rows = len(frozen_original)
    expected_columns = frozen_original.shape[1]

    freeze_validation_rows.append({
        'dataset_id': dataset_id,
        'csv_rows_ok': len(csv_reload) == expected_rows,
        'csv_columns_ok': csv_reload.shape[1] == expected_columns,
        'parquet_rows_ok': len(parquet_reload) == expected_rows,
        'parquet_columns_ok': (
            parquet_reload.shape[1] == expected_columns
        ),
        'record_ids_unique_csv': csv_reload['record_id'].is_unique,
        'record_ids_unique_parquet': (
            parquet_reload['record_id'].is_unique
        ),
        'groups_consistent_csv': bool(
            (
                csv_reload.groupby('feature_fingerprint_sha256')[
                    'duplicate_group_id'
                ].nunique()
                == 1
            ).all()
        ),
        'groups_consistent_parquet': bool(
            (
                parquet_reload.groupby('feature_fingerprint_sha256')[
                    'duplicate_group_id'
                ].nunique()
                == 1
            ).all()
        ),
    })

freeze_validation = pd.DataFrame(freeze_validation_rows)
freeze_validation.to_csv(
    RESULTS_DIR / '09_freeze_validation.csv',
    index=False,
    encoding='utf-8-sig',
)

if not freeze_validation.drop(
    columns=['dataset_id']
).all(axis=None):
    raise RuntimeError(
        'Falló al menos una comprobación posterior al congelamiento.'
    )

print('Validación posterior al congelamiento: CORRECTA')
display(freeze_validation)


## 13. Crear diccionario inicial de variables

In [ ]:

scale_behavior = {
    # Dry Bean
    'Area': 'quadratic_with_scale',
    'Perimeter': 'linear_with_scale',
    'MajorAxisLength': 'linear_with_scale',
    'MinorAxisLength': 'linear_with_scale',
    'AspectRation': 'scale_invariant',
    'Eccentricity': 'scale_invariant',
    'ConvexArea': 'quadratic_with_scale',
    'EquivDiameter': 'linear_with_scale',
    'Extent': 'scale_invariant',
    'Solidity': 'scale_invariant',
    'roundness': 'scale_invariant',
    'Compactness': 'scale_invariant',
    'ShapeFactor1': 'inverse_linear_with_scale',
    'ShapeFactor2': 'requires_definition_review',
    'ShapeFactor3': 'scale_invariant',
    'ShapeFactor4': 'requires_definition_review',
    # INIAP Rice
    'AREA': 'quadratic_with_scale',
    'PERIMETER': 'linear_with_scale',
    'MAJOR_AXIS': 'linear_with_scale',
    'MINOR_AXIS': 'linear_with_scale',
    'ECCENTRICITY': 'scale_invariant',
    'CONVEX_AREA': 'quadratic_with_scale',
    'EXTENT': 'scale_invariant',
}

algebraic_status = {
    'AspectRation': 'exactly_reconstructible',
    'Eccentricity': 'exactly_reconstructible',
    'EquivDiameter': 'exactly_reconstructible',
    'Solidity': 'exactly_reconstructible',
    'roundness': 'exactly_reconstructible',
    'Compactness': 'exactly_reconstructible',
    'ShapeFactor1': 'exactly_reconstructible',
    'ShapeFactor3': 'exactly_reconstructible',
    'ECCENTRICITY': 'approximately_reconstructible_due_to_rounding',
}

dictionary_rows = []

for dataset_id, dataframe in datasets_raw.items():
    target = dataset_columns[dataset_id]['target']

    for column in dataframe.columns:
        dictionary_rows.append({
            'dataset_id': dataset_id,
            'variable': column,
            'role': 'target' if column == target else 'feature',
            'dtype_observed': str(dataframe[column].dtype),
            'unique_values': int(
                dataframe[column].nunique(dropna=True)
            ),
            'missing_values': int(
                dataframe[column].isna().sum()
            ),
            'pixel_scale_behavior': (
                'not_applicable_target'
                if column == target
                else scale_behavior.get(
                    column,
                    'requires_definition_review',
                )
            ),
            'algebraic_status': (
                'not_applicable_target'
                if column == target
                else algebraic_status.get(
                    column,
                    'not_verified_as_exactly_reconstructible',
                )
            ),
        })

data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(
    PROTOCOL_DIR / 'initial_data_dictionary.csv',
    index=False,
    encoding='utf-8-sig',
)

display(data_dictionary)


## 14. Generar resumen reproducible de la ejecución

In [ ]:

run_summary = {
    'notebook': '01_auditoria_y_congelamiento_Grain_Robustness_Q1',
    'run_timestamp_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'random_seed': RANDOM_SEED,
    'project_root': str(PROJECT_ROOT),
    'python_version': sys.version,
    'platform': platform.platform(),
    'cpu_count': os.cpu_count(),
    'package_versions': {
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'datasets': {},
}

for dataset_id, dataframe in datasets_raw.items():
    target = dataset_columns[dataset_id]['target']
    run_summary['datasets'][dataset_id] = {
        'rows': len(dataframe),
        'columns': dataframe.shape[1],
        'features': len(dataset_columns[dataset_id]['features']),
        'classes': int(dataframe[target].nunique()),
        'class_counts': {
            str(label): int(count)
            for label, count in dataframe[
                target
            ].value_counts().items()
        },
        'source_sha256': source_manifest.loc[
            source_manifest['dataset_id'] == dataset_id,
            'sha256',
        ].iloc[0],
        'exact_duplicate_rows_involved': int(
            dataframe.duplicated(
                subset=dataframe.columns.tolist(),
                keep=False,
            ).sum()
        ),
    }

summary_path = RESULTS_DIR / 'audit_freeze_run_summary.json'
summary_path.write_text(
    json.dumps(
        run_summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(json.dumps(run_summary, indent=2, ensure_ascii=False))


## 15. Empaquetar los resultados

In [ ]:

zip_path_local = (
    PROJECT_ROOT
    / '07_results'
    / 'NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip'
)

if zip_path_local.exists():
    zip_path_local.unlink()

paths_to_package = [
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    FROZEN_DIR,
    PROTOCOL_DIR / 'initial_data_dictionary.csv',
]

with zipfile.ZipFile(
    zip_path_local,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for item in paths_to_package:
        if item.is_file():
            archive.write(
                item,
                arcname=item.relative_to(PROJECT_ROOT),
            )
        elif item.is_dir():
            for file_path in item.rglob('*'):
                if file_path.is_file():
                    archive.write(
                        file_path,
                        arcname=file_path.relative_to(PROJECT_ROOT),
                    )


def copy_file_to_drive_with_retry(
    source,
    destination,
    attempts=8,
    delay_seconds=5,
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            shutil.copy2(source, destination)

            if not destination.exists():
                raise IOError(
                    f'No aparece el archivo de destino: {destination}'
                )

            if destination.stat().st_size != source.stat().st_size:
                raise IOError(
                    'El tamaño del archivo copiado no coincide.'
                )

            return True

        except (
            ConnectionAbortedError,
            ConnectionResetError,
            OSError,
            IOError,
        ) as error:
            last_error = error
            print(
                f'Sincronización {attempt}/{attempts} falló para '
                f'{source.name}: {type(error).__name__}: {error}'
            )
            time.sleep(delay_seconds)

    print(
        'ADVERTENCIA: no se pudo sincronizar este archivo con Drive:',
        source,
        'Último error:',
        last_error,
    )
    return False


# Sincronizar los resultados finales archivo por archivo.
sync_rows = []

for source in PROJECT_ROOT.rglob('*'):
    if not source.is_file():
        continue

    relative = source.relative_to(PROJECT_ROOT)
    destination = DRIVE_PROJECT_ROOT / relative

    success = copy_file_to_drive_with_retry(
        source,
        destination,
    )

    sync_rows.append({
        'source': str(source),
        'destination': str(destination),
        'size_bytes': source.stat().st_size,
        'synchronized': success,
    })

sync_report = pd.DataFrame(sync_rows)
sync_report_local = (
    PROJECT_ROOT
    / '07_results'
    / 'drive_sync_report.csv'
)
sync_report.to_csv(
    sync_report_local,
    index=False,
    encoding='utf-8-sig',
)

# Copiar también el reporte de sincronización.
copy_file_to_drive_with_retry(
    sync_report_local,
    DRIVE_PROJECT_ROOT
    / '07_results'
    / 'drive_sync_report.csv',
)

zip_path_drive = (
    DRIVE_PROJECT_ROOT
    / '07_results'
    / 'NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip'
)

success_count = int(sync_report['synchronized'].sum())
failure_count = int((~sync_report['synchronized']).sum())

print('Ejecución completada correctamente.')
print('Datos congelados localmente:', FROZEN_DIR)
print('Resultados locales:', RESULTS_DIR)
print('ZIP local:', zip_path_local)
print('ZIP esperado en Drive:', zip_path_drive)
print('Archivos sincronizados:', success_count)
print('Archivos no sincronizados:', failure_count)

if failure_count > 0:
    print(
        '\nIMPORTANTE: el análisis terminó y el ZIP local está completo, '
        'pero Drive presentó fallos durante la sincronización. '
        'Puedes descargar manualmente el ZIP desde el panel Archivos de '
        f'Colab: {zip_path_local}'
    )
else:
    print(
        '\nSincronización con Google Drive completada correctamente.'
    )



## Resultado esperado

El notebook debe finalizar mostrando:

```text
Validación posterior al congelamiento: CORRECTA
Ejecución completada correctamente.
Sincronización con Google Drive completada correctamente.
```

El ZIP persistente debe quedar en:

```text
/content/drive/MyDrive/Grain_Robustness_Q1/07_results/
NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip
```

También se conserva una copia local en:

```text
/content/Grain_Robustness_Q1_runtime/07_results/
NOTEBOOK01_AUDIT_FREEZE_OUTPUTS.zip
```

Si Google Drive vuelve a desconectarse durante la sincronización, el análisis
no se pierde: descarga la copia local desde el panel **Archivos** de Colab.

Ese ZIP será la entrada para el Notebook 02, dedicado a redundancia
estructural y construcción formal de las representaciones.
